# Laboratorios de Pandas: limpieza, análisis y combinación de datos de Instacart

**Aprendiz:** Carlos Eduardo Herrera Sibaja
**Programa:** Análisis y Desarrollo de Software
**Evidencia:** GA1-220501094-01-AA1-EV06

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este cuaderno desarrolla, en orden, las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python*. Cada lección tiene tres partes:

1. **Ejemplos de la lección:** las técnicas ejecutadas sobre los datos de Instacart.
2. **Laboratorio guiado:** un caso completo que combina esas técnicas.
3. **Reto:** siete ejercicios resueltos con una interpretación breve.

Las lecciones comparten variables, así que el cuaderno debe ejecutarse completo y en orden (**Run All**). Los datos crudos se leen de `data/raw/` y los resultados se guardan en `data/processed/`.

## 0. Preparación del entorno y carga de los datos

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(parents=True, exist_ok=True)

print('pandas', pd.__version__)

pandas 3.1.0


Se cargan las cinco tablas con `sep=';'`, igual que en la actividad anterior (`data_quality.ipynb`). Estas variables son el punto de partida de la Lección 2.

In [2]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(f'{nombre:<12} {df.shape}')

orders       (478967, 6)
order_prods  (4545007, 4)
products     (49694, 4)
aisles       (134, 2)
departments  (21, 2)


---
# Lección 2: Limpieza y estandarización de datos crudos

Orden de trabajo: duplicados, tipos, nulos, texto, duplicados implícitos y verificación. Para cada problema se decide si **eliminar**, **corregir** o **conservar**, según lo que significa el dato.

## 2.1 Ejemplos de la lección

### Duplicados exactos: detectar con `duplicated` y eliminar con `drop_duplicates`

In [3]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


Las 30 filas son 15 pedidos y sus 15 copias, todas del miércoles a las 2 a. m.: un fallo de carga, no pedidos reales. Es seguro eliminarlas.

In [4]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


### Tipos de dato con `astype`

In [5]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB
Después: 7.2 MB


### Nulos: comprobar la causa antes de decidir

In [6]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


Cada nulo de `days_since_prior_order` es un primer pedido: no hay pedido anterior. Se **conserva** el nulo; rellenarlo con 0 inventaría una recompra el mismo día.

In [7]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


| Columna | Causa | Decisión |
|---|---|---|
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo (pasillo 100 y departamento 21, `missing`) | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

### Texto con los métodos `.str`

In [8]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16
0 0


### Duplicados implícitos

In [9]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141


268 de 280


Los nombres repetidos tienen `product_id` distintos y casi todos aparecen en pedidos. Se **conservan y documentan**; en los análisis se agrupa por `product_id`, no por nombre.

### Limpieza reutilizable con funciones

In [10]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)

Verificación de toda la limpieza (si la celda no muestra errores, la limpieza está completa):

In [11]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Limpieza verificada')

Limpieza verificada


## 2.2 Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Objetivo: una secuencia de pasos de limpieza encadenados que parte de los archivos crudos, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso.

### Paso 1: recargar los datos crudos

In [12]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


### Paso 2: una función que anota el estado de la tabla

In [13]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

### Paso 3: un paso de limpieza por función

In [14]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

### Paso 4: encadenar los pasos con `pipe`

In [15]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

### Paso 5: el informe de cambios

In [16]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación:** quitar duplicados eliminó 15 filas de `orders` y, con ellas, 2 nulos (dos de los pedidos repetidos eran primeros pedidos). Corregir tipos no cambió filas ni nulos, pero bajó la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB (−63 %). En `products` los 1 258 nulos de nombre pasaron a `'Unknown'`. Los 836 nulos de `order_prods` se conservan a propósito.

### Paso 6: validar el resultado

In [17]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


### Paso 7: guardar las tablas limpias y el informe en `data/processed/`

Parquet conserva los tipos (`Int16`, `int8`, etc.) y ocupa mucho menos que CSV. El informe de cambios se guarda en CSV para poder abrirlo en Excel.

In [18]:
orders_limpio.to_parquet(SALIDA / 'orders_clean.parquet', index=False)
products_limpio.to_parquet(SALIDA / 'products_clean.parquet', index=False)
op_limpio.to_parquet(SALIDA / 'order_products_clean.parquet', index=False)
informe.to_csv(SALIDA / 'cleaning_report.csv', index=False)

for archivo in sorted(SALIDA.glob('*')):
    print(f'{archivo.name:<32} {archivo.stat().st_size / 1e6:6.1f} MB')

.gitkeep                            0.0 MB
cleaning_report.csv                 0.0 MB
order_products_clean.parquet       32.5 MB
orders_clean.parquet                5.4 MB
products_clean.parquet              1.5 MB


**Conclusiones clave del laboratorio 2**
- Un pipeline es una cadena de funciones pequeñas que reciben y devuelven un DataFrame, unidas con `pipe`.
- La función `anotar` intercalada entre pasos produce el informe de cambios sin esfuerzo extra.
- La validación con `assert` convierte las reglas de calidad en una prueba que se ejecuta cada vez.

## 2.3 Reto 2

**Ejercicio 1.** ¿`order_prods` tiene filas duplicadas exactas? ¿Se repite algún par `order_id` + `product_id`? ¿`reordered` solo contiene 0 y 1?

In [19]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:',
      order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))

Filas duplicadas exactas: 0


Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]


**Interpretación:** `order_prods` no tiene filas duplicadas exactas ni pares `order_id` + `product_id` repetidos: cada producto aparece una sola vez por pedido, así que la clave compuesta es única. `reordered` solo toma 0 y 1, como corresponde a una bandera. No requiere limpieza.

**Ejercicio 2.** ¿`aisles` y `departments` tienen nombres duplicados o con espacios al inicio o al final? ¿Necesitan limpieza de texto?

In [20]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    serie = df[col]
    print(nombre)
    print('  duplicados exactos:            ', serie.duplicated().sum())
    print('  duplicados al normalizar:      ', serie.str.strip().str.lower().duplicated().sum())
    print('  con espacios al inicio o final:', (serie != serie.str.strip()).sum())
    print('  con espacios dobles:           ', serie.str.contains('  ', regex=False).sum())

aisles
  duplicados exactos:             0
  duplicados al normalizar:       0
  con espacios al inicio o final: 0
  con espacios dobles:            0
departments
  duplicados exactos:             0
  duplicados al normalizar:       0
  con espacios al inicio o final: 0
  con espacios dobles:            0


**Interpretación:** ninguno de los dos catálogos tiene nombres duplicados (ni siquiera al ignorar mayúsculas y espacios) ni espacios sobrantes. No necesitan limpieza de texto; la única particularidad es la categoría `missing`, que se trata en el ejercicio 7.

**Ejercicio 3.** Los 836 nulos de `add_to_cart_order`: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno? ¿Qué los explica?

In [21]:
nulos_carrito = order_prods['add_to_cart_order'].isna()
pedidos_con_nulos = order_prods.loc[nulos_carrito, 'order_id'].unique()
tamano_pedido = order_prods.groupby('order_id').size()

print('Nulos:', nulos_carrito.sum())
print('Pedidos afectados:', len(pedidos_con_nulos))
print('Mínimo de productos en esos pedidos:', tamano_pedido.loc[pedidos_con_nulos].min())
print('Posición máxima registrada en el carrito:', order_prods['add_to_cart_order'].max())

# ¿Los nulos de cada pedido son exactamente los productos que pasan de la posición 64?
nulos_por_pedido = order_prods[nulos_carrito].groupby('order_id').size()
print('Nulos = productos - 64 en todos los pedidos:',
      (nulos_por_pedido == tamano_pedido.loc[nulos_por_pedido.index] - 64).all())

Nulos: 836
Pedidos afectados: 70


Mínimo de productos en esos pedidos: 65
Posición máxima registrada en el carrito: 64
Nulos = productos - 64 en todos los pedidos: True


**Interpretación:** los 836 nulos están en 70 pedidos, todos con 65 productos o más, y la posición máxima registrada es 64. En cada pedido, los nulos son exactamente los productos que pasan de la posición 64. El sistema solo registraba 64 posiciones del carrito: el producto existe y el nulo significa "posición no registrada". Se conserva el nulo, porque inventar una posición falsearía la posición media.

**Ejercicio 4.** Cargar de nuevo `products.csv` en `products_crudo` y contar cuántos nombres (no nulos) cambiaron con la estandarización.

In [22]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
con_nombre = products_crudo['product_name'].notna()

cambiaron = products_crudo.loc[con_nombre, 'product_name'] != products.loc[con_nombre, 'product_name']
print('Nombres no nulos:', con_nombre.sum())
print('Nombres que cambiaron:', cambiaron.sum(), f'({cambiaron.mean():.1%})')

ejemplos = pd.DataFrame({'original': products_crudo.loc[con_nombre, 'product_name'][cambiaron],
                         'estandarizado': products.loc[con_nombre, 'product_name'][cambiaron]})
ejemplos.head(5)

Nombres no nulos: 48436
Nombres que cambiaron: 6321 (13.1%)


,original,estandarizado
17,Pizza for One Suprema Frozen Pizza,Pizza For One Suprema Frozen Pizza
23,Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...,Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29,Three Cheese Ziti- Marinara with Meatballs,Three Cheese Ziti- Marinara With Meatballs
35,Traditional Lasagna with Meat Sauce Savory Ita...,Traditional Lasagna With Meat Sauce Savory Ita...
41,Biotin 1000 mcg,Biotin 1000 Mcg


**Interpretación:** 6 321 de los 48 436 nombres no nulos (13,1 %) cambiaron. La mayoría de los cambios son de mayúsculas (`for` → `For`, `mcg` → `Mcg`) y una parte son espacios dobles o invisibles. Estandarizar evita que el mismo producto escrito de dos formas se cuente como dos.

**Ejercicio 5.** Crear una Series con el nombre del día de cada pedido, convertirla a categoría ordenada de domingo a sábado y comparar la memoria del texto con la de la categoría.

In [23]:
dias = ['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']

nombre_dia = orders['order_dow'].map(dict(enumerate(dias)))
dia_categoria = pd.Series(pd.Categorical(nombre_dia, categories=dias, ordered=True),
                          index=nombre_dia.index, name='dia')

mem_texto = nombre_dia.memory_usage(deep=True) / 1e6
mem_categoria = dia_categoria.memory_usage(deep=True) / 1e6
print(f'Texto:     {mem_texto:.2f} MB ({nombre_dia.dtype})')
print(f'Categoría: {mem_categoria:.2f} MB ({dia_categoria.dtype})')
print(f'Reducción: {1 - mem_categoria / mem_texto:.1%}')
print('Orden respetado:', dia_categoria.min(), '→', dia_categoria.max())

Texto:     7.14 MB (str)
Categoría: 0.48 MB (category)
Reducción: 93.3%
Orden respetado: domingo → sábado


**Interpretación:** la Series de texto ocupa 7,14 MB y la categoría 0,48 MB, un 93 % menos, porque la categoría guarda cada nombre una sola vez y cada fila solo almacena un código pequeño. Además, al ser ordenada, `min()` y `max()` respetan el calendario (domingo → sábado) en lugar del orden alfabético.

**Ejercicio 6.** ¿Cuántas filas de `order_prods` corresponden a productos `'Unknown'` y en cuántos pedidos aparecen? ¿Se deberían eliminar?

In [24]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
es_unknown = order_prods['product_id'].isin(ids_unknown)

print('Filas con productos Unknown:', es_unknown.sum(), f'({es_unknown.mean():.2%} del total)')
print('Pedidos en que aparecen:', order_prods.loc[es_unknown, 'order_id'].nunique())
print('Pedidos en que son el único producto:',
      (order_prods.groupby('order_id')['product_id']
                  .apply(lambda s: s.isin(ids_unknown).all())).sum())

Filas con productos Unknown: 9547 (0.21% del total)


Pedidos en que aparecen: 8254


Pedidos en que son el único producto: 24


**Interpretación:** 9 547 filas (0,21 % de los artículos) corresponden a productos `'Unknown'` y aparecen en 8 254 pedidos; en 24 de ellos son el único producto. No conviene eliminarlas: son compras reales, se perderían artículos de más de 8 000 pedidos y 24 pedidos quedarían vacíos, lo que alteraría el tamaño de los pedidos y la tasa de recompra. Basta con identificarlas y excluirlas solo cuando el análisis dependa del nombre o la categoría del producto.

**Ejercicio 7.** En una copia de `departments`, reemplazar `'missing'` por un nulo real (`pd.NA`) y explicar por qué no conviene hacerlo en la tabla original antes de agrupar.

In [25]:
departments_copia = departments.copy()
departments_copia['department'] = departments_copia['department'].replace('missing', pd.NA)
print(departments_copia.isna().sum())

# Efecto al agrupar: groupby descarta las claves nulas por defecto
productos_por_depto = products.merge(departments_copia, on='department_id', how='left')
print('Grupos con "missing" como texto:', products.merge(departments, on='department_id')
      .groupby('department').ngroups)
print('Grupos con pd.NA:', productos_por_depto.groupby('department').ngroups)
print('Productos que desaparecen del resumen:',
      len(products) - productos_por_depto.groupby('department').size().sum())

department_id    0
department       1
dtype: int64
Grupos con "missing" como texto: 21
Grupos con pd.NA: 20
Productos que desaparecen del resumen: 1258


**Interpretación:** con `pd.NA`, `groupby` descarta la clave nula por defecto: el resumen pasa de 21 a 20 departamentos y los 1 258 productos sin catálogo desaparecen sin aviso. Por eso no conviene cambiarlo en la tabla original antes de agrupar: `'missing'` como texto mantiene esos productos visibles como un grupo propio y los totales siguen cuadrando.

---
# Lección 3: Filtrado, ordenamiento y agrupación de datos

Cada pregunta se traduce primero a la forma **"métrica por grupo"** o a una **condición**. Por ejemplo, "¿a qué hora se compra más?" se convierte en "número de pedidos por hora".

## 3.1 Ejemplos de la lección

### Filtrado con máscaras booleanas

In [26]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


In [27]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

45599


### Filtrado con `query`

In [28]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

1449


### Ordenamiento con `sort_values` y `nlargest`

In [29]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100
        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


### Agrupación con `groupby`

In [30]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

10 40578


In [31]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


### Tabla dinámica con `pivot_table`

In [32]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


### Valores por fila con `transform`

In [33]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')

231946 30527


## 3.2 Laboratorio guiado 3: perfil de hábitos de compra

Encargo de mercadeo: diseñar campañas según qué tan seguido compra cada cliente y saber a qué hora compra cada tipo de cliente. Plan: contar pedidos por cliente, segmentar clientes, clasificar pedidos en franjas horarias y cruzar segmentos con franjas.

### Paso 1: pedidos por cliente

In [34]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

### Paso 2: segmentar clientes con `pd.cut`

In [35]:
segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


### Paso 3: llevar el segmento a cada pedido con `map`

In [36]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


### Paso 4: resumen por segmento

In [37]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


**Interpretación:** el 19,4 % de los clientes (los frecuentes) hace casi la mitad de los pedidos (231 946 de 478 952) y vuelve con una mediana de 6 días; los regulares vuelven cada 11 días y los ocasionales cada 16. Los ocasionales tienen espera porque su único pedido en estos datos no es necesariamente el primero que hicieron en la tienda.

### Paso 5: franjas horarias y tabla cruzada

In [38]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


**Interpretación:** todos los segmentos compran sobre todo en la tarde, pero los frecuentes compran más en la mañana (35,2 % frente a 30,0 % de los ocasionales) y algo menos en la tarde y la noche. Para mercadeo, una campaña de fidelización dirigida a clientes frecuentes debería enviarse por la mañana.

### Paso 6: las combinaciones de día y franja más activas

In [39]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

### Paso 7: guardar los resúmenes y dejar `orders` como estaba

In [40]:
resumen_dia.to_csv(SALIDA / 'orders_by_weekday.csv')
resumen_segmento.to_csv(SALIDA / 'customer_segments.csv')
perfil.to_csv(SALIDA / 'segment_time_profile.csv')

orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

(478952, 6)


**Conclusiones clave del laboratorio 3**
- Una pregunta de negocio compleja se resuelve en pasos: agrupar, categorizar, devolver al detalle y cruzar.
- `pd.cut` convierte números en categorías con límites explícitos.
- `map` lleva un resultado calculado por grupo a cada fila del detalle.
- `crosstab` con `normalize` permite comparar grupos de distinto tamaño.

## 3.3 Reto 3

**Ejercicio 1.** ¿Cuáles son las tres horas con más pedidos los domingos y cuántos pedidos tiene cada una?

*Condición + métrica por grupo:* pedidos con `order_dow == 0` → número de pedidos por hora → las tres mayores.

In [41]:
domingos = orders[orders['order_dow'] == 0]
domingos.groupby('order_hour_of_day').size().nlargest(3).rename('pedidos')

order_hour_of_day
15    7585
14    7533
13    7483
Name: pedidos, dtype: int64

**Interpretación:** los domingos el pico es a primera hora de la tarde: 15 h (7 585 pedidos), 14 h (7 533) y 13 h (7 483). Las tres horas tienen volúmenes casi iguales, así que el domingo la demanda alta es una franja continua, más tardía que el pico general de la semana (10 a. m.).

**Ejercicio 2.** ¿A qué hora es mayor la espera media entre pedidos y a qué hora es menor?

*Métrica por grupo:* media de `days_since_prior_order` por hora.

In [42]:
espera_por_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean()
print(f'Mayor espera: {espera_por_hora.idxmax()} h ({espera_por_hora.max():.2f} días)')
print(f'Menor espera: {espera_por_hora.idxmin()} h ({espera_por_hora.min():.2f} días)')

Mayor espera: 0 h (12.52 días)
Menor espera: 22 h (10.62 días)


**Interpretación:** la mayor espera media está a las 0 h (12,52 días) y la menor a las 22 h (10,62 días). La diferencia es de unos 2 días: quien compra a medianoche tiende a ser un cliente menos regular, mientras que las compras de las 22 h vienen de clientes que vuelven más seguido.

**Ejercicio 3.** ¿Cuántos clientes tienen 10 pedidos o más?

*Métrica por grupo + condición:* número de pedidos por cliente ≥ 10.

In [43]:
pedidos_por_cliente = orders.groupby('user_id').size()
print('Clientes con 10 pedidos o más:', (pedidos_por_cliente >= 10).sum())

Clientes con 10 pedidos o más: 6327


**Interpretación:** 6 327 clientes tienen 10 pedidos o más, apenas el 4 % de los 157 437 clientes. Es un núcleo pequeño de clientes muy fieles, útil para estudiar hábitos de recompra.

**Ejercicio 4.** ¿Cuántos pedidos se hicieron exactamente 7 días después del anterior y qué porcentaje del total representan?

*Condición:* `days_since_prior_order == 7`.

In [44]:
a_los_7 = (orders['days_since_prior_order'] == 7).sum()
print('Pedidos a los 7 días:', a_los_7)
print(f'Porcentaje del total: {a_los_7 / len(orders):.2%}')
print('Valor más frecuente de espera:', orders['days_since_prior_order'].mode()[0], 'días')

Pedidos a los 7 días: 44577
Porcentaje del total: 9.31%
Valor más frecuente de espera: 30 días


**Interpretación:** 44 577 pedidos (9,31 % del total) se hicieron exactamente una semana después del anterior. Es uno de los picos de la distribución: muchos clientes hacen la compra semanal el mismo día. El valor más frecuente es 30 días, pero ese pico está inflado porque 30 es el tope de la columna (agrupa "30 días o más").

**Ejercicio 5.** Usando `order_prods`: ¿cuántos pedidos tienen 20 productos o más? ¿Cuál es el pedido más grande y cuántos productos tiene?

*Métrica por grupo:* número de productos por pedido.

In [45]:
productos_por_pedido = order_prods.groupby('order_id').size()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum())
print('Pedido más grande:', productos_por_pedido.idxmax(),
      'con', productos_por_pedido.max(), 'productos')

Pedidos con 20 productos o más: 49420


Pedido más grande: 61355 con 127 productos


**Interpretación:** 49 420 pedidos tienen 20 productos o más (el 11 % de los 450 046 pedidos con productos). El pedido más grande es el 61355, con 127 productos, muy por encima del límite de 64 posiciones del carrito que explica los nulos del Reto 2.

**Ejercicio 6.** Usando `order_prods`: tasa de recompra (media de `reordered`) para las posiciones 1 a 5 del carrito. ¿Qué patrón se ve?

*Condición + métrica por grupo:* posiciones 1 a 5 → media de `reordered` por posición.

In [46]:
primeras_posiciones = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
primeras_posiciones.groupby('add_to_cart_order')['reordered'].mean().round(3).rename('tasa_recompra')

add_to_cart_order
1    0.679
2    0.677
3    0.659
4    0.638
5    0.617
Name: tasa_recompra, dtype: float64

**Interpretación:** la tasa de recompra baja a medida que avanza el carrito: 0,679 en la posición 1 y 0,617 en la 5. Los clientes agregan primero los productos que ya conocen y compran siempre, y dejan para después los productos nuevos o de prueba.

**Ejercicio 7.** Comparar la mediana de `days_since_prior_order` de los pedidos de fin de semana (domingo y sábado) con la del resto de la semana.

*Métrica por grupo:* mediana de la espera según la máscara `isin([0, 6])`.

In [47]:
fin_de_semana = orders['order_dow'].isin([0, 6]).map({True: 'fin de semana', False: 'entre semana'})
orders.groupby(fin_de_semana)['days_since_prior_order'].agg(['median', 'mean', 'count']).round(2)

,median,mean,count
order_dow,,,
entre semana,7.0,10.87,312767
fin de semana,8.0,11.62,137368


**Interpretación:** la mediana de espera es de 8 días para los pedidos de fin de semana y de 7 días entre semana (medias de 11,62 y 10,87). Quienes compran entre semana vuelven un poco antes, lo que encaja con la compra semanal en un día laboral fijo.

---
# Lección 4: Combinación y estructuración de múltiples datasets

Se unen las cinco tablas en una **tabla maestra** donde cada fila es un producto dentro de un pedido, con su nombre, pasillo, departamento y los datos del pedido. Un `merge` mal hecho no da error: multiplica o pierde filas en silencio, por eso cada unión se valida.

## 4.1 Ejemplos de la lección

### Tipos de unión con el parámetro `how`

In [48]:
pedidos = pd.DataFrame({'order_id': [1, 2, 3], 'user_id': [10, 20, 30]})
detalle = pd.DataFrame({'order_id': [1, 1, 2, 4], 'product_id': [100, 200, 100, 300]})

for tipo in ['inner', 'left', 'outer']:
    resultado = pedidos.merge(detalle, on='order_id', how=tipo)
    print(tipo, len(resultado))
    print(resultado, '\n')

inner 3
   order_id  user_id  product_id
0         1       10         100
1         1       10         200
2         2       20         100 

left 4
   order_id  user_id  product_id
0         1       10       100.0
1         1       10       200.0
2         2       20       100.0
3         3       30         NaN 

outer 5
   order_id  user_id  product_id
0         1     10.0       100.0
1         1     10.0       200.0
2         2     20.0       100.0
3         3     30.0         NaN
4         4      NaN       300.0 



### Validación de la relación con `validate`

In [49]:
catalogo_malo = pd.concat([products, products.head(1)])

try:
    order_prods.merge(catalogo_malo, on='product_id', validate='m:1')
except pd.errors.MergeError as error:
    print('Detectado:', str(error).splitlines()[0])

Detectado: Merge keys are not unique in right dataset; not a many-to-one merge


### Auditoría de la unión con `indicator`

In [50]:
auditoria = orders.merge(order_prods[['order_id']].drop_duplicates(),
                         on='order_id', how='left', indicator=True)
auditoria['_merge'].value_counts()

_merge
both          450046
left_only      28906
right_only         0
Name: count, dtype: int64

### Construcción de la tabla maestra

In [51]:
filas_inicio = len(order_prods)

maestra = order_prods.merge(products, on='product_id', how='left', validate='m:1')
print('+ products   ', maestra.shape)
maestra = maestra.merge(aisles, on='aisle_id', how='left', validate='m:1')
print('+ aisles     ', maestra.shape)
maestra = maestra.merge(departments, on='department_id', how='left', validate='m:1')
print('+ departments', maestra.shape)
maestra = maestra.merge(orders, on='order_id', how='left', validate='m:1')
print('+ orders     ', maestra.shape)

assert len(maestra) == filas_inicio
assert maestra[['product_name', 'aisle', 'department', 'user_id']].notna().all().all()

+ products    (4545007, 7)


+ aisles      (4545007, 8)


+ departments (4545007, 9)


+ orders      (4545007, 14)


### Preguntas sobre la tabla maestra

In [52]:
top = (maestra.groupby(['product_id', 'product_name']).size()
              .nlargest(5).reset_index(name='veces'))
print(top)

por_depto = (maestra.groupby('department')
                    .agg(articulos=('product_id', 'size'),
                         tasa_recompra=('reordered', 'mean'))
                    .sort_values('articulos', ascending=False))
por_depto.head(3)

   product_id            product_name  veces
0       24852                  Banana  66050
1       13176  Bag Of Organic Bananas  53297
2       21137    Organic Strawberries  37039
3       21903    Organic Baby Spinach  33971
4       47209    Organic Hass Avocado  29773


,articulos,tasa_recompra
department,,
produce,1328302,0.651140
dairy eggs,759842,0.670597
snacks,404850,0.574442


### Apilado con `concat`

In [53]:
categorias = pd.concat([
    aisles.rename(columns={'aisle_id': 'id', 'aisle': 'nombre'}).assign(tipo='pasillo'),
    departments.rename(columns={'department_id': 'id', 'department': 'nombre'}).assign(tipo='departamento'),
], ignore_index=True)
print(categorias.shape)

(155, 3)


### Formato ancho y largo con `pivot_table` y `melt`

In [54]:
ancho = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                           values='order_id', aggfunc='count')
largo = (ancho.reset_index()
              .melt(id_vars='order_hour_of_day', var_name='order_dow', value_name='pedidos'))
print(ancho.shape, largo.shape)

(24, 7) (168, 3)


### Guardado del resultado

In [55]:
maestra.to_parquet(SALIDA / 'instacart_master.parquet', index=False)
por_depto.to_csv(SALIDA / 'department_summary.csv')
largo.to_csv(SALIDA / 'orders_by_hour_weekday_long.csv', index=False)

## 4.2 Laboratorio guiado 4: tabla de análisis por producto

Encargo del equipo de compras: una tabla con una fila por producto (veces pedido, clientes, tasa de recompra y posición media en el carrito) para encontrar el producto más fiel de cada departamento.

### Paso 1: agregar la tabla maestra por producto

In [56]:
por_producto = (maestra
    .groupby('product_id')
    .agg(veces=('order_id', 'size'),
         clientes=('user_id', 'nunique'),
         tasa_recompra=('reordered', 'mean'),
         posicion_media=('add_to_cart_order', 'mean'))
    .reset_index())
print(por_producto.shape)
por_producto.head(3)

(45573, 5)


,product_id,veces,clientes,tasa_recompra,posicion_media
0,1,280,206,0.564286,5.642857
1,2,11,11,0.000000,8.727273
2,3,42,27,0.738095,6.952381


### Paso 2: recuperar nombre y departamento con `merge`

In [57]:
ficha_productos = (por_producto
    .merge(products, on='product_id', how='left', validate='1:1')
    .merge(departments, on='department_id', how='left', validate='m:1')
    [['product_id', 'product_name', 'department', 'veces', 'clientes',
      'tasa_recompra', 'posicion_media']])
print(ficha_productos.shape)
ficha_productos.loc[ficha_productos['veces'].idxmax()]

(45573, 7)


product_id           24852
product_name        Banana
department         produce
veces                66050
clientes             33274
tasa_recompra     0.844254
posicion_media    4.874815
Name: 22808, dtype: object

### Paso 3: filtrar los productos con suficientes datos

Una tasa de recompra calculada con pocos pedidos no es confiable; se conservan los productos pedidos al menos 1 000 veces.

In [58]:
populares = ficha_productos[ficha_productos['veces'] >= 1000].copy()
print(len(populares))

739


### Paso 4: clasificar dentro de cada departamento con `rank`

In [59]:
populares['puesto'] = (populares
    .groupby('department')['tasa_recompra']
    .rank(ascending=False, method='first'))

lideres = (populares[populares['puesto'] == 1]
           .sort_values('tasa_recompra', ascending=False))
print(len(lideres))
lideres[['department', 'product_name', 'veces', 'tasa_recompra']].head(5)

18


,department,product_name,veces,tasa_recompra
41704,dairy eggs,Whole Organic Omega 3 Milk,1286,0.863142
22808,produce,Banana,66050,0.844254
29030,snacks,Extra Fancy Unsalted Mixed Nuts,1436,0.822423
42302,beverages,Zero Calorie Cola,1256,0.818471
3570,bakery,Organic Honey Sweet Whole Wheat Bread,1607,0.799004


**Interpretación:** el producto más pedido no siempre es el más fiel. En lácteos, la leche Whole Organic Omega 3 Milk tiene pocos pedidos (1 286), pero el 86,3 % de las veces es una recompra; la Banana, con 66 050 pedidos, queda segunda. Para el equipo de compras, estos 18 líderes son productos con demanda estable que no deberían faltar en inventario.

### Paso 5: la misma respuesta con `sort_values` y `head`

In [60]:
lideres_2 = (populares
    .sort_values('tasa_recompra', ascending=False)
    .groupby('department')
    .head(1))
print(len(lideres_2), set(lideres_2['product_id']) == set(lideres['product_id']))

18 True


### Paso 6: guardar la ficha de productos

In [61]:
ficha_productos.to_parquet(SALIDA / 'product_profile.parquet', index=False)
lideres.to_csv(SALIDA / 'department_leaders.csv', index=False)

**Conclusiones clave del laboratorio 4**
- Agregar con `groupby` cambia el nivel de la tabla: de artículos a productos, clientes o días.
- Después de agregar, un `merge` con `validate='1:1'` recupera las columnas descriptivas.
- Filtrar por un mínimo de casos evita conclusiones basadas en pocos datos.
- `groupby().rank()` y `sort_values().groupby().head()` resuelven "el mejor de cada grupo".

## 4.3 Reto 4

Todos los ejercicios usan la tabla `maestra`.

**Ejercicio 1.** Entre los pasillos con al menos 10 000 artículos pedidos, ¿cuáles son los tres con mayor tasa de recompra? ¿Qué pasillo tiene la tasa más baja de todos?

In [62]:
por_pasillo = (maestra.groupby('aisle')
                      .agg(articulos=('product_id', 'size'),
                           tasa_recompra=('reordered', 'mean')))
grandes = por_pasillo[por_pasillo['articulos'] >= 10000]

print('Pasillos con al menos 10 000 artículos:', len(grandes), 'de', len(por_pasillo))
print('\nTop 3 por tasa de recompra:')
print(grandes.nlargest(3, 'tasa_recompra').round(3))
print('\nTasa más baja de todos los pasillos:')
print(por_pasillo.nsmallest(1, 'tasa_recompra').round(3))

Pasillos con al menos 10 000 artículos: 77 de 134

Top 3 por tasa de recompra:
                               articulos  tasa_recompra
aisle                                                  
milk                              125522          0.781
water seltzer sparkling water     117133          0.731
fresh fruits                      510101          0.718

Tasa más baja de todos los pasillos:
                   articulos  tasa_recompra
aisle                                      
spices seasonings      29454          0.152


**Interpretación:** 77 de los 134 pasillos tienen al menos 10 000 artículos. Los de mayor recompra son `milk` (0,781), `water seltzer sparkling water` (0,731) y `fresh fruits` (0,718): productos básicos que se reponen en cada compra. La tasa más baja de todos es la de `spices seasonings` (0,152): las especias duran mucho y se compran pocas veces.

**Ejercicio 2.** Comparar la participación de cada departamento en los pedidos de madrugada (0 a 5 h) con su participación total. ¿Qué departamento entra en el top 3 de madrugada pero no en el total?

In [63]:
madrugada = maestra[maestra['order_hour_of_day'].between(0, 5)]

participacion = pd.DataFrame({
    'madrugada_pct': madrugada['department'].value_counts(normalize=True),
    'total_pct': maestra['department'].value_counts(normalize=True),
}).mul(100).round(2)
participacion['diferencia'] = (participacion['madrugada_pct'] - participacion['total_pct']).round(2)

top3_madrugada = set(participacion['madrugada_pct'].nlargest(3).index)
top3_total = set(participacion['total_pct'].nlargest(3).index)
print('Top 3 madrugada:', sorted(top3_madrugada))
print('Top 3 total:    ', sorted(top3_total))
print('En el top 3 de madrugada pero no en el total:', sorted(top3_madrugada - top3_total))
participacion.sort_values('madrugada_pct', ascending=False).head(6)

Top 3 madrugada: ['beverages', 'dairy eggs', 'produce']
Top 3 total:     ['dairy eggs', 'produce', 'snacks']
En el top 3 de madrugada pero no en el total: ['beverages']


,madrugada_pct,total_pct,diferencia
department,,,
produce,30.21,29.23,0.98
dairy eggs,15.87,16.72,-0.85
beverages,8.49,8.26,0.23
snacks,7.89,8.91,-1.02
frozen,7.65,6.91,0.74
pantry,6.17,5.78,0.39


**Interpretación:** `beverages` entra en el top 3 de madrugada (8,49 %) pero no en el total, donde lo supera `snacks`. En la madrugada ganan peso `produce` (+0,98 puntos) y `frozen` (+0,74), y lo pierden `snacks` (−1,02) y `dairy eggs` (−0,85). Las diferencias son de alrededor de un punto porcentual, así que la mezcla de compras de madrugada es muy parecida a la del resto del día.

**Ejercicio 3.** ¿Cuáles son los tres productos más pedidos del departamento `snacks`?

In [64]:
(maestra[maestra['department'] == 'snacks']
    .groupby(['product_id', 'product_name']).size()
    .nlargest(3)
    .reset_index(name='veces'))

,product_id,product_name,veces
0,13870,Lightly Salted Baked Snap Pea Crisps,3462
1,34448,Original Veggie Straws,2331
2,31040,Sea Salt Pita Chips,2275


**Interpretación:** los snacks más pedidos son Lightly Salted Baked Snap Pea Crisps (3 462), Original Veggie Straws (2 331) y Sea Salt Pita Chips (2 275). Los tres son snacks horneados o de vegetales, en línea con la preferencia por productos orgánicos y saludables que se ve en el top general.

**Ejercicio 4.** Tabla con una fila por cliente y tres columnas: pedidos distintos, artículos y tasa de recompra. ¿Cuántos clientes tiene, cuál es el promedio de artículos por cliente y qué cliente compró más artículos?

In [65]:
por_cliente = (maestra.groupby('user_id')
                      .agg(pedidos=('order_id', 'nunique'),
                           articulos=('product_id', 'size'),
                           tasa_recompra=('reordered', 'mean')))
print('Clientes:', len(por_cliente))
print(f"Promedio de artículos por cliente: {por_cliente['articulos'].mean():.1f}")
print('Cliente con más artículos:', por_cliente['articulos'].idxmax(),
      'con', por_cliente['articulos'].max(), 'artículos')
por_cliente.loc[[por_cliente['articulos'].idxmax()]]

Clientes: 149626
Promedio de artículos por cliente: 30.4
Cliente con más artículos: 103634 con 580 artículos


,pedidos,articulos,tasa_recompra
user_id,,,
103634,18,580,0.822414


**Interpretación:** la tabla tiene 149 626 clientes (los que tienen al menos un pedido con productos) y cada uno compró en promedio 30,4 artículos. El cliente 103634 es el que más compró: 580 artículos en 18 pedidos, con una tasa de recompra de 0,82, es decir, un comprador muy fiel a sus productos.

**Ejercicio 5.** Unir `orders` con los pedidos de `order_prods` usando `how='inner'`. ¿Cuántas filas se obtienen y por qué son menos que con `how='left'`?

In [66]:
pedidos_con_productos = order_prods[['order_id']].drop_duplicates()

union_inner = orders.merge(pedidos_con_productos, on='order_id', how='inner', validate='1:1')
union_left = orders.merge(pedidos_con_productos, on='order_id', how='left',
                          validate='1:1', indicator=True)

print('inner:', len(union_inner))
print('left: ', len(union_left))
print('Diferencia:', len(union_left) - len(union_inner))
print(union_left['_merge'].value_counts())

inner: 450046
left:  478952
Diferencia: 28906
_merge
both          450046
left_only      28906
right_only         0
Name: count, dtype: int64


**Interpretación:** con `inner` se obtienen 450 046 filas y con `left` 478 952: la diferencia son los 28 906 pedidos (6 %) que no tienen productos registrados en `order_prods`. `inner` solo conserva las claves presentes en ambas tablas y los descarta sin aviso; `left` los mantiene y `indicator=True` los muestra como `left_only`.

**Ejercicio 6.** Con `concat`, una sola tabla con los tres productos más pedidos de `produce` y los tres de `dairy eggs`, con una columna que indique el departamento.

In [67]:
def top_productos(departamento, n=3):
    return (maestra[maestra['department'] == departamento]
            .groupby(['product_id', 'product_name']).size()
            .nlargest(n)
            .reset_index(name='veces')
            .assign(department=departamento))

top_produce_dairy = pd.concat([top_productos('produce'), top_productos('dairy eggs')],
                              ignore_index=True)
top_produce_dairy

,product_id,product_name,veces,department
0,24852,Banana,66050,produce
1,13176,Bag Of Organic Bananas,53297,produce
2,21137,Organic Strawberries,37039,produce
3,27845,Organic Whole Milk,19600,dairy eggs
4,49235,Organic Half & Half,10752,dairy eggs
5,27086,Half & Half,9588,dairy eggs


**Interpretación:** en `produce` dominan las bananas y las fresas, con volúmenes muy superiores a los de lácteos: el tercer producto de `produce` (37 039) casi duplica al primero de `dairy eggs` (Organic Whole Milk, 19 600). En lácteos, Half & Half aparece dos veces, en versión orgánica y convencional, porque son productos distintos con su propio `product_id`.

**Ejercicio 7.** Tabla dinámica de artículos por departamento (filas) y día de la semana (columnas). ¿Qué forma tiene y qué día se piden más artículos de `alcohol`?

In [68]:
articulos_depto_dia = maestra.pivot_table(index='department', columns='order_dow',
                                          values='product_id', aggfunc='count')
print('Forma:', articulos_depto_dia.shape)
alcohol = articulos_depto_dia.loc['alcohol']
print('Artículos de alcohol por día:', alcohol.to_dict())
print('Día con más artículos de alcohol:', alcohol.idxmax(), f'({dias[alcohol.idxmax()]})')

articulos_depto_dia.to_csv(SALIDA / 'items_by_department_weekday.csv')

Forma: (21, 7)
Artículos de alcohol por día: {0: 2397, 1: 3025, 2: 2868, 3: 3094, 4: 3656, 5: 3814, 6: 2787}
Día con más artículos de alcohol: 5 (viernes)


**Interpretación:** la tabla tiene forma (21, 7): 21 departamentos por 7 días. El alcohol se compra más el viernes (día 5, 3 814 artículos) y el jueves (3 656), y menos el domingo (2 397). Es el único departamento con el pico fuera del domingo o el lunes (los otros 20 tienen su máximo en uno de esos dos días): el alcohol se compra antes del fin de semana.

---
# Archivos generados

Resumen de los archivos guardados en `data/processed/`:

In [69]:
for archivo in sorted(SALIDA.glob('*')):
    print(f'{archivo.name:<36} {archivo.stat().st_size / 1e6:7.2f} MB')

.gitkeep                                0.00 MB
cleaning_report.csv                     0.00 MB
customer_segments.csv                   0.00 MB
department_leaders.csv                  0.00 MB
department_summary.csv                  0.00 MB
instacart_master.parquet              103.62 MB
items_by_department_weekday.csv         0.00 MB
order_products_clean.parquet           32.51 MB
orders_by_hour_weekday_long.csv         0.00 MB
orders_by_weekday.csv                   0.00 MB
orders_clean.parquet                    5.39 MB
product_profile.parquet                 1.69 MB
products_clean.parquet                  1.45 MB
segment_time_profile.csv                0.00 MB


# Conclusiones generales

- **Lección 2 (limpieza):** los datos crudos tenían 15 pedidos duplicados, 1 258 productos sin nombre y nombres sin estandarizar. Con un pipeline documentado se eliminaron los duplicados, se rellenaron los nombres con `'Unknown'`, se estandarizó el texto y se ajustaron los tipos, con una reducción de memoria de hasta el 69 %. Los nulos de `days_since_prior_order` y `add_to_cart_order` se conservaron porque tienen una causa conocida.
- **Lección 3 (análisis):** la demanda se concentra los domingos y lunes, entre las 10 y las 15 h. El 19,4 % de los clientes (frecuentes) hace casi la mitad de los pedidos, vuelve cada 6 días y compra más por la mañana. La recompra es mayor en los primeros productos del carrito.
- **Lección 4 (combinación):** la tabla maestra une las cinco tablas sin perder ni duplicar filas (4 545 007), validada con `validate` e `indicator`. Con ella se ve que los productos básicos (leche, agua, fruta) son los más recomprados y que el producto más fiel de cada departamento no siempre es el más vendido.

Respuesta a la pregunta orientadora: los datos de Instacart quedan confiables cuando cada problema de calidad se trata según su causa (eliminar, corregir o conservar), cada paso queda registrado y validado, y cada unión entre tablas se comprueba. Sobre esa base, los hábitos de compra se pueden describir con cifras reproducibles.